In [2]:
from astropy.io import fits
import numpy as np

# ============================================================
# FILE PATHS
# ============================================================
fits_files = {
    "Y": "/Users/aishwarya/Desktop/new_cdfs/trimmed_2deg/trim2deg_y_sci.fits",
    "Z": "/Users/aishwarya/Desktop/new_cdfs/trimmed_2deg/trim2deg_z_sci.fits",
    "I": "/Users/aishwarya/Desktop/new_cdfs/trimmed_2deg/trim2deg_i_sci.fits"
}

# ============================================================
# PARAMETERS
# ============================================================
PIXEL_SCALE = 0.27  # arcsec/pixel

# ============================================================
# LOAD DATA
# ============================================================
data = {}
for band, path in fits_files.items():
    with fits.open(path) as hdul:
        data[band] = hdul[0].data.astype(float)

# ============================================================
# FUNCTION: COMPUTE AREA FROM MASK
# ============================================================
def compute_area(mask):
    N_pixels = np.sum(mask)
    area_arcsec2 = N_pixels * (PIXEL_SCALE ** 2)
    area_deg2 = area_arcsec2 / (3600.0 ** 2)
    return N_pixels, area_deg2

# ============================================================
# CREATE MASKS (ONLY NaNs REMOVED)
# ============================================================
masks = {}
for band in ["Y", "Z", "I"]:
    masks[band] = np.isfinite(data[band])   # <-- KEY CHANGE

# ============================================================
# AREA PER BAND
# ============================================================
print("=== AREA PER BAND ===")
for band in ["Y", "Z", "I"]:
    N, area = compute_area(masks[band])
    print(f"{band}-band: Pixels = {N:,} | Area = {area:.5f} deg^2")

# ============================================================
# COMMON AREA (intersection of all 3)
# ============================================================
common_mask = masks["Y"] & masks["Z"] & masks["I"]

N_common, area_common = compute_area(common_mask)

print("\n=== COMMON USABLE AREA (Y ∩ Z ∩ I) ===")
print(f"Pixels = {N_common:,}")
print(f"Area   = {area_common:.5f} deg^2")

=== AREA PER BAND ===
Y-band: Pixels = 376,184,737 | Area = 2.11604 deg^2
Z-band: Pixels = 376,183,437 | Area = 2.11603 deg^2
I-band: Pixels = 376,183,437 | Area = 2.11603 deg^2

=== COMMON USABLE AREA (Y ∩ Z ∩ I) ===
Pixels = 374,287,356
Area   = 2.10537 deg^2


In [1]:
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
import astropy.units as u
import numpy as np

# Load mask
file_name = "/Users/aishwarya/Desktop/Mask/cdfs_mask.fits"
with fits.open(file_name) as hdu:
    data = hdu[0].data
    wcs = WCS(hdu[0].header)

# Pixel area in deg²
pixel_scales = proj_plane_pixel_scales(wcs) * u.deg
pix_area_deg = pixel_scales[0] * pixel_scales[1]

# Count good pixels
good_pixels = data > 0.5  # assuming 1 = good
n_good_pixels = np.sum(good_pixels)

# Total area in deg²
total_area_deg2 = n_good_pixels * pix_area_deg
print(f"Total area of good pixels: {total_area_deg2:.4f}")

Total area of good pixels: 2.1156 deg2


In [2]:
from astropy.io import fits
import numpy as np

# ============================================================
# FILE PATHS
# ============================================================
FILES = {
    "I": {
        "img": "/Users/aishwarya/Desktop/Lyman_alpha_2/Mosaiced_images/Trim2deg/trim2deg_mosaic_i.fits",
        "wgt": "/Users/aishwarya/Desktop/Lyman_alpha_2/Mosaiced_images/Trim2deg/trim2deg_weight_i.fits",
        "wmin": 0.0008
    },
    "Z": {
        "img": "/Users/aishwarya/Desktop/Lyman_alpha_2/Mosaiced_images/Trim2deg/trim2deg_mosaic_z.fits",
        "wgt": "/Users/aishwarya/Desktop/Lyman_alpha_2/Mosaiced_images/Trim2deg/trim2deg_weight_z.fits",
        "wmin": 0.0008
    },
    "Y": {
        "img": "/Users/aishwarya/Desktop/Lyman_alpha_2/Y_band/Trim/trim2deg_mosaic_y.fits",
        "wgt": "/Users/aishwarya/Desktop/Lyman_alpha_2/Y_band/Trim/trim2deg_weight_y.fits",
        "wmin": 0.001
    }
}

PIXEL_SCALE = 0.27  # arcsec/pixel

# ============================================================
# LOAD DATA
# ============================================================
data = {}
weights = {}

for band in FILES:
    with fits.open(FILES[band]["img"]) as hdul:
        data[band] = hdul[0].data.astype(float)

    with fits.open(FILES[band]["wgt"]) as hdul:
        weights[band] = hdul[0].data.astype(float)

# ============================================================
# FUNCTION: AREA
# ============================================================
def compute_area(mask):
    N_pixels = np.sum(mask)
    area_arcsec2 = N_pixels * (PIXEL_SCALE ** 2)
    area_deg2 = area_arcsec2 / (3600.0 ** 2)
    return N_pixels, area_deg2

# ============================================================
# BUILD MASKS (SCIENCE + WEIGHT)
# ============================================================
masks = {}

for band in FILES:
    sci_mask = np.isfinite(data[band])        # DS9 + trimming mask
    wgt_mask = weights[band] > FILES[band]["wmin"]  # depth cut

    masks[band] = sci_mask & wgt_mask

# ============================================================
# AREA PER BAND
# ============================================================
print("=== AREA PER BAND (after masking + weight cut) ===")
for band in ["I", "Z", "Y"]:
    N, area = compute_area(masks[band])
    print(f"{band}-band: Pixels = {N:,} | Area = {area:.5f} deg^2")

# ============================================================
# COMMON AREA (I ∩ Z ∩ Y)
# ============================================================
common_mask = masks["I"] & masks["Z"] & masks["Y"]

N_common, area_common = compute_area(common_mask)

print("\n=== COMMON USABLE AREA (I ∩ Z ∩ Y) ===")
print(f"Pixels = {N_common:,}")
print(f"Area   = {area_common:.5f} deg^2")

=== AREA PER BAND (after masking + weight cut) ===
I-band: Pixels = 556,074,998 | Area = 3.12792 deg^2
Z-band: Pixels = 564,614,074 | Area = 3.17595 deg^2
Y-band: Pixels = 566,917,716 | Area = 3.18891 deg^2

=== COMMON USABLE AREA (I ∩ Z ∩ Y) ===
Pixels = 554,609,868
Area   = 3.11968 deg^2
